# Contact Center Anomalies & EDA

Exploratory data analysis on Supabase contact/phone records.

**Sections:**
1. Connect & load data
2. Data quality audit
3. Wait-time & SLA outliers
4. Abandonment hotspots
5. Agent behavior anomalies
6. Volume & pattern anomalies
7. Actionable findings summary

Baselines use **all data**; anomalies flagged against **last 7 and 30 days**.

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from supabase import create_client

# Load .env from project root
env_path = Path(__file__).resolve().parent.parent / '.env' if '__file__' in dir() else Path.cwd().parent / '.env'
load_dotenv(env_path)

SUPABASE_URL = os.getenv('VITE_SUPABASE_URL')
SUPABASE_KEY = os.getenv('VITE_SUPABASE_PUBLISHABLE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    # Fallback: try loading from the project root directly
    load_dotenv(Path.cwd().parent / '.env')
    SUPABASE_URL = os.getenv('VITE_SUPABASE_URL')
    SUPABASE_KEY = os.getenv('VITE_SUPABASE_PUBLISHABLE_KEY')

assert SUPABASE_URL and SUPABASE_KEY, 'Missing VITE_SUPABASE_URL or VITE_SUPABASE_PUBLISHABLE_KEY in .env'
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

# Authenticate (RLS requires a logged-in session)
AUTH_EMAIL = os.getenv('SUPABASE_AUTH_EMAIL', 'team@company.com')
AUTH_PASS = os.getenv('SUPABASE_AUTH_PASSWORD')
if AUTH_PASS:
    sb.auth.sign_in_with_password({'email': AUTH_EMAIL, 'password': AUTH_PASS})
    print(f'Authenticated as {AUTH_EMAIL}')
else:
    print('Warning: SUPABASE_AUTH_PASSWORD not set in .env. RLS may block data access.')
    print('Add SUPABASE_AUTH_PASSWORD to your .env file to authenticate.')

print(f'Connected to {SUPABASE_URL}')

In [ ]:
def fetch_all(table: str, page_size: int = 1000) -> list:
    """Fetch all rows from a Supabase table with pagination."""
    all_rows = []
    frm = 0
    while True:
        resp = sb.table(table).select('*').range(frm, frm + page_size - 1).execute()
        data = resp.data
        if not data:
            break
        all_rows.extend(data)
        if len(data) < page_size:
            break
        frm += page_size
    return all_rows

print('Loading contacts...')
contacts_raw = fetch_all('contacts')
print(f'  {len(contacts_raw)} contacts loaded')

print('Loading phone_records...')
phones_raw = fetch_all('phone_records')
print(f'  {len(phones_raw)} phone records loaded')

df = pd.DataFrame(contacts_raw)
phones = pd.DataFrame(phones_raw)

# Convert timestamp columns to datetime
ts_cols = [
    'initiation_timestamp', 'enqueue_timestamp', 'connected_to_agent_timestamp',
    'disconnect_timestamp', 'acw_start_timestamp', 'acw_end_timestamp',
    'scheduled_timestamp'
]
for col in ts_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Convert numeric columns
num_cols = ['agent_connection_attempts', 'number_of_holds', 'contact_duration', 'agent_interaction_duration']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Derived columns
df['date'] = df['initiation_timestamp'].dt.date
df['hour'] = df['initiation_timestamp'].dt.hour
df['dow'] = df['initiation_timestamp'].dt.dayofweek  # 0=Mon
df['dow_name'] = df['initiation_timestamp'].dt.day_name()

def shift_of(h):
    if pd.isna(h): return None
    h = int(h)
    if 6 <= h < 14: return '1st'
    if 14 <= h < 22: return '2nd'
    return '3rd'

df['shift'] = df['hour'].apply(shift_of)
df['is_answered'] = df['connected_to_agent_timestamp'].notna()
df['is_abandoned'] = ~df['is_answered']

# Time-diff calculations (seconds)
df['connect_sec'] = (df['connected_to_agent_timestamp'] - df['initiation_timestamp']).dt.total_seconds()
df['handle_sec'] = (df['disconnect_timestamp'] - df['initiation_timestamp']).dt.total_seconds()
df['acw_sec'] = (df['acw_end_timestamp'] - df['acw_start_timestamp']).dt.total_seconds()
df['interaction_sec'] = (df['disconnect_timestamp'] - df['connected_to_agent_timestamp']).dt.total_seconds()
df['queue_sec'] = (df['connected_to_agent_timestamp'] - df['enqueue_timestamp']).dt.total_seconds()
# Inclusive wait time (for abandoned contacts, time from initiation to disconnect)
df['wait_sec'] = (df['disconnect_timestamp'] - df['initiation_timestamp']).dt.total_seconds()

# Week start for grouping
df['week'] = df['initiation_timestamp'].dt.to_period('W').apply(lambda p: p.start_time.date())

# Merge phone descriptions
if 'system_phone_number' in df.columns and 'phone_number' in phones.columns:
    phone_lookup = phones.set_index('phone_number')['description'].to_dict()
    df['phone_description'] = df['system_phone_number'].map(phone_lookup).fillna(df.get('phone_description', pd.Series(dtype=str)))

print(f'\nDataFrame shape: {df.shape}')
print(f'Date range: {df["initiation_timestamp"].min()} — {df["initiation_timestamp"].max()}')
print(f'Answered: {df["is_answered"].sum()} | Abandoned: {df["is_abandoned"].sum()}')
df.head()

---
## 2. Data Quality Audit

Missing values, out-of-order timestamps, field distributions.

In [ ]:
# Null rate per column
null_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('Null rate per column (%):')
print(null_pct.to_string())
print()

# Timestamp sequence validation
# For answered contacts, expected order:
# initiation -> enqueue -> connected_to_agent -> disconnect -> acw_start -> acw_end
answered = df[df['is_answered']].copy()
seq_violations = pd.Series(dtype=int)

checks = [
    ('initiation > enqueue', answered['initiation_timestamp'] > answered['enqueue_timestamp']),
    ('enqueue > connected', answered['enqueue_timestamp'] > answered['connected_to_agent_timestamp']),
    ('connected > disconnect', answered['connected_to_agent_timestamp'] > answered['disconnect_timestamp']),
    ('disconnect > acw_start', answered['disconnect_timestamp'] > answered['acw_start_timestamp']),
    ('acw_start > acw_end', answered['acw_start_timestamp'] > answered['acw_end_timestamp']),
]

print('Timestamp sequence violations (answered contacts):')
for label, mask in checks:
    count = mask.sum()
    pct = count / len(answered) * 100 if len(answered) > 0 else 0
    print(f'  {label}: {count} ({pct:.1f}%)')
    if count > 0:
        seq_violations[label] = count
print()

# Agent connection attempts distribution
if 'agent_connection_attempts' in df.columns:
    print('Agent connection attempts distribution:')
    print(df['agent_connection_attempts'].value_counts().sort_index().to_string())
    print()
    multi = (df['agent_connection_attempts'] > 1).sum()
    print(f'Contacts with >1 connection attempt: {multi} ({multi/len(df)*100:.1f}%)')

print()

# Holds distribution
if 'number_of_holds' in df.columns:
    print('Number of holds distribution:')
    print(df['number_of_holds'].value_counts().sort_index().to_string())

In [ ]:
# Key distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Connect time distribution (capped at 99th percentile for readability)
ax = axes[0, 0]
connect = df['connect_sec'].dropna()
cap = connect.quantile(0.99)
connect_capped = connect[connect <= cap]
ax.hist(connect_capped, bins=60, color='#3B82F6', alpha=0.8)
ax.axvline(connect.median(), color='#EF4444', linestyle='--', label=f'Median: {connect.median():.0f}s')
ax.axvline(20, color='#F59E0B', linestyle=':', label='SLA 20s')
ax.axvline(60, color='#DC2626', linestyle=':', label='SLA 60s')
ax.set_title('Connect Time Distribution (capped at p99)')
ax.legend()
ax.set_xlabel('Seconds')

# Handle time distribution
ax = axes[0, 1]
handle = df['handle_sec'].dropna() / 60  # to minutes
handle_cap = handle.quantile(0.99)
handle_capped = handle[handle <= handle_cap]
ax.hist(handle_capped, bins=60, color='#22C55E', alpha=0.8)
ax.axvline(handle.median(), color='#EF4444', linestyle='--', label=f'Median: {handle.median():.1f}min')
ax.set_title('Handle Time Distribution (capped at p99)')
ax.legend()
ax.set_xlabel('Minutes')

# ACW time distribution
ax = axes[1, 0]
acw = df['acw_sec'].dropna() / 60
acw_cap = acw.quantile(0.99)
acw_capped = acw[acw <= acw_cap]
ax.hist(acw_capped, bins=60, color='#8B5CF6', alpha=0.8)
ax.axvline(acw.median(), color='#EF4444', linestyle='--', label=f'Median: {acw.median():.1f}min')
ax.set_title('ACW Time Distribution (capped at p99)')
ax.legend()
ax.set_xlabel('Minutes')

# Queue time distribution
ax = axes[1, 1]
queue = df['queue_sec'].dropna()
queue_cap = queue.quantile(0.99)
queue_capped = queue[queue <= queue_cap]
ax.hist(queue_capped, bins=60, color='#F97316', alpha=0.8)
ax.axvline(queue.median(), color='#EF4444', linestyle='--', label=f'Median: {queue.median():.0f}s')
ax.axvline(60, color='#DC2626', linestyle=':', label='SLA 60s')
ax.set_title('Queue Time Distribution (capped at p99)')
ax.legend()
ax.set_xlabel('Seconds')

plt.tight_layout()
plt.show()

---
## 3. Wait-Time & SLA Outliers

Z-scores on connect/handle/queue times by agent, queue, and shift.
Flag entities where the last 7/30 days deviate > 2σ from the overall baseline.

In [ ]:
now = df['initiation_timestamp'].max()
last_7d = now - pd.Timedelta(days=7)
last_30d = now - pd.Timedelta(days=30)
df_7 = df[df['initiation_timestamp'] >= last_7d]
df_30 = df[df['initiation_timestamp'] >= last_30d]
print(f'Last 7 days: {len(df_7)} contacts')
print(f'Last 30 days: {len(df_30)} contacts')
print(f'All time: {len(df)} contacts')
print(f'Date range: {df["initiation_timestamp"].min()} — {now}')

In [ ]:
def z_outliers(df_all, df_recent, group_col, metric_col, min_group=10, z_thresh=2):
    """
    Compute z-scores: compare each group's recent metric mean to the
    overall population mean and std. Flag groups > z_thresh.
    """
    # Overall baseline
    baseline_mean = df_all[metric_col].dropna().mean()
    baseline_std = df_all[metric_col].dropna().std()
    if baseline_std == 0 or pd.isna(baseline_std):
        return pd.DataFrame()

    # Recent group stats
    recent = (
        df_recent
        .groupby(group_col)[metric_col]
        .agg(['mean', 'count'])
        .reset_index()
    )
    recent.columns = [group_col, 'recent_mean', 'recent_count']
    recent = recent[recent['recent_count'] >= min_group]
    recent['z_score'] = (recent['recent_mean'] - baseline_mean) / baseline_std
    recent['baseline_mean'] = baseline_mean
    recent['baseline_std'] = baseline_std
    recent['abs_z'] = recent['z_score'].abs()
    recent = recent.sort_values('abs_z', ascending=False)
    return recent[recent['abs_z'] >= z_thresh]

metrics_config = [
    ('connect_sec', 'Avg Connect Time (s)'),
    ('handle_sec', 'Avg Handle Time (s)'),
    ('queue_sec', 'Avg Queue Time (s)'),
    ('acw_sec', 'Avg ACW Time (s)'),
]

group_cols = ['agent', 'queue', 'shift']

all_outliers = []
for group_col in group_cols:
    for metric_col, metric_name in metrics_config:
        # Z-score against ALL data baseline
        out_7 = z_outliers(df, df_7, group_col, metric_col)
        out_30 = z_outliers(df, df_30, group_col, metric_col)

        for period, out_df in [('7d', out_7), ('30d', out_30)]:
            if out_df.empty:
                continue
            for _, row in out_df.iterrows():
                all_outliers.append({
                    'period': period,
                    'group': group_col,
                    'entity': row[group_col],
                    'metric': metric_name,
                    'recent_mean': round(row['recent_mean'], 1),
                    'baseline_mean': round(row['baseline_mean'], 1),
                    'z_score': round(row['z_score'], 2),
                    'count': int(row['recent_count']),
                })

outliers_df = pd.DataFrame(all_outliers)
if outliers_df.empty:
    print('No significant wait-time outliers found (no groups with |z| >= 2)')
else:
    outliers_df = outliers_df.sort_values('z_score', ascending=False, key=abs)
    print(f'Found {len(outliers_df)} z-score outliers (|z| >= 2)')
    print(f'  Last 7d: {(outliers_df["period"] == "7d").sum()}')
    print(f'  Last 30d: {(outliers_df["period"] == "30d").sum()}')
    print()
    print('Top 20 outliers:')
outliers_df.head(20)

In [ ]:
# SLA threshold analysis by group
def sla_table(df_in, group_col, thresholds=[20, 60, 120]):
    """Compute SLA % at each threshold, grouped by group_col."""
    answered = df_in[df_in['is_answered']].copy()
    groups = []
    for name, grp in answered.groupby(group_col):
        row = {'entity': name, 'total': len(grp)}
        for t in thresholds:
            met = (grp['connect_sec'] <= t).sum()
            row[f'sla_{t}s'] = round(met / len(grp) * 100, 1) if len(grp) > 0 else 0
        groups.append(row)
    return pd.DataFrame(groups).sort_values('total', ascending=False)

for group_col in ['agent', 'queue', 'shift']:
    print(f'\nSLA by {group_col}:')
    sla_all = sla_table(df, group_col)
    sla_7 = sla_table(df_7, group_col)
    sla_30 = sla_table(df_30, group_col)

    # Compare recent vs baseline for biggest drops
    if not sla_all.empty and not sla_7.empty:
        merged = sla_all [['entity', 'total', 'sla_20s', 'sla_60s']].merge(
            sla_7[['entity', 'sla_20s', 'sla_60s']],
            on='entity', how='inner', suffixes=('_all', '_7d')
        )
        merged['sla_60s_drop'] = merged['sla_60s_7d'] - merged['sla_60s_all']
        merged = merged.sort_values('sla_60s_drop')
        print(f'  Biggest SLA≤60s drops in last 7d vs baseline:')
        for _, r in merged.head(5).iterrows():
            print(f'    {r["entity"]}: {r["sla_60s_all"]:.1f}% (all) → {r["sla_60s_7d"]:.1f}% (7d) [{r["sla_60s_drop"]:+.1f} pts]')
    print()

In [ ]:
# Visualize connect time distributions by shift
fig = go.Figure()
for s in ['1st', '2nd', '3rd']:
    subset = df[df['shift'] == s]['connect_sec'].dropna()
    cap = subset.quantile(0.98)
    subset = subset[subset <= cap]
    fig.add_trace(go.Violin(
        y=subset,
        name=f'{s} shift (n={len(subset)})',
        box_visible=True,
        meanline_visible=True,
        opacity=0.7,
    ))
fig.update_layout(
    title='Connect Time Distribution by Shift',
    yaxis_title='Seconds',
    height=500,
)
fig.show()

---
## 4. Abandonment Hotspots

Rates by queue, shift, hour-of-day, and day-of-week.
Disconnect reason distribution and recent vs. baseline comparison.

In [ ]:
# Overall abandonment rate
total = len(df)
abandoned = df['is_abandoned'].sum()
answered = df['is_answered'].sum()
print(f'Overall: {abandoned:,} abandoned / {total:,} total = {abandoned/total*100:.1f}%')
print(f'Last 7d:  {df_7["is_abandoned"].sum():,} / {len(df_7):,} = {df_7["is_abandoned"].sum()/len(df_7)*100:.1f}%' if len(df_7) > 0 else 'No data in last 7d')
print(f'Last 30d: {df_30["is_abandoned"].sum():,} / {len(df_30):,} = {df_30["is_abandoned"].sum()/len(df_30)*100:.1f}%' if len(df_30) > 0 else 'No data in last 30d')
print()

# Abandonment by queue
def abandon_rate(df_in, group_col):
    grp = df_in.groupby(group_col).agg(
        total=('is_abandoned', 'count'),
        abandoned=('is_abandoned', 'sum'),
    ).reset_index()
    grp['rate'] = (grp['abandoned'] / grp['total'] * 100).round(1)
    return grp.sort_values('rate', ascending=False)

queue_abandon_all = abandon_rate(df, 'queue')
queue_abandon_7 = abandon_rate(df_7, 'queue') if len(df_7) > 0 else pd.DataFrame()
queue_abandon_30 = abandon_rate(df_30, 'queue') if len(df_30) > 0 else pd.DataFrame()

print('Abandonment rate by queue (all time):')
print(queue_abandon_all.to_string(index=False))
print()

if not queue_abandon_7.empty:
    # Compare 7d vs all-time
    merged = queue_abandon_all[['queue', 'rate']].merge(
        queue_abandon_7[['queue', 'rate']].rename(columns={'rate': 'rate_7d'}),
        on='queue', how='inner'
    )
    merged['delta'] = merged['rate_7d'] - merged['rate']
    merged = merged.sort_values('delta', ascending=False)
    print('Abandonment rate delta (7d vs baseline):')
    for _, r in merged.head(10).iterrows():
        print(f'  {r["queue"]}: {r["rate"]:.1f}% (all) → {r["rate_7d"]:.1f}% (7d) [{r["delta"]:+.1f} pts]')
    print()

In [ ]:
# Abandonment by shift and hour-of-day
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By shift
ax = axes[0]
shift_abandon = abandon_rate(df, 'shift')
shifts_order = ['1st', '2nd', '3rd']
shift_abandon = shift_abandon.set_index('shift').reindex(shifts_order).reset_index()
bars = ax.bar(shift_abandon['shift'], shift_abandon['rate'], color=['#3B82F6', '#F97316', '#8B5CF6'])
for bar, row in zip(bars, shift_abandon.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.rate}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Abandonment Rate by Shift')
ax.set_ylabel('Abandonment Rate (%)')
ax.set_ylim(0, max(shift_abandon['rate'].max() * 1.3, 5))

# By hour
ax = axes[1]
hour_abandon = df.groupby('hour').agg(
    total=('is_abandoned', 'count'),
    abandoned=('is_abandoned', 'sum'),
).reset_index()
hour_abandon['rate'] = (hour_abandon['abandoned'] / hour_abandon['total'] * 100).round(1)
ax.bar(hour_abandon['hour'], hour_abandon['rate'], color='#3B82F6', alpha=0.7)
ax.set_title('Abandonment Rate by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Abandonment Rate (%)')
ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

In [ ]:
# Disconnect reason analysis
if 'disconnect_reason' in df.columns:
    reasons = df['disconnect_reason'].fillna('(none)').value_counts()
    print('Disconnect reason distribution:')
    print(reasons.to_string())
    print()

    # Disconnect reasons for abandoned contacts only
    abandoned_df = df[df['is_abandoned']]
    if len(abandoned_df) > 0:
        abandon_reasons = abandoned_df['disconnect_reason'].fillna('(none)').value_counts()
        print('Disconnect reasons for ABANDONED contacts:')
        print(abandon_reasons.to_string())
        print()

        # Top queues by each disconnect reason
        if 'queue' in df.columns:
            print('Top queues by abandon disconnect reason:')
            for reason in abandon_reasons.head(3).index:
                subset = abandoned_df[abandoned_df['disconnect_reason'] == reason]
                top_queues = subset['queue'].value_counts().head(3)
                print(f'  {reason}:')
                for q, c in top_queues.items():
                    print(f'    {q}: {c}')
    print()

    # Disconnect reason shift by time period
    if len(df_7) > 10:
        reason_all = df['disconnect_reason'].fillna('(none)').value_counts(normalize=True) * 100
        reason_7 = df_7['disconnect_reason'].fillna('(none)').value_counts(normalize=True) * 100
        reason_compare = pd.DataFrame({'all_pct': reason_all, '7d_pct': reason_7}).fillna(0)
        reason_compare['delta'] = (reason_compare['7d_pct'] - reason_compare['all_pct']).round(1)
        reason_compare = reason_compare.sort_values('delta', ascending=False)
        print('Disconnect reason share changes (7d vs all):')
        print(reason_compare.head(10).to_string())

---
## 5. Agent Behavior Anomalies

- Agent connection attempts > 1
- Holds per contact
- Excessive ACW time
- Repeat contacts from same customer

In [ ]:
# Agent-level metrics scoreboard
answered_df = df[df['is_answered']].copy()

agent_stats = answered_df.groupby('agent').agg(
    total_contacts=('contact_id', 'count'),
    avg_connect_sec=('connect_sec', 'mean'),
    median_connect_sec=('connect_sec', 'median'),
    avg_handle_sec=('handle_sec', 'mean'),
    avg_acw_sec=('acw_sec', 'mean'),
    avg_holds=('number_of_holds', 'mean'),
    avg_attempts=('agent_connection_attempts', 'mean'),
    pct_multi_attempt=('agent_connection_attempts', lambda x: (x > 1).mean() * 100),
    pct_sla_20s=('connect_sec', lambda x: (x <= 20).mean() * 100),
    pct_sla_60s=('connect_sec', lambda x: (x <= 60).mean() * 100),
).reset_index()

# Compute z-scores for agent connect time vs overall
overall_mean_connect = answered_df['connect_sec'].mean()
overall_std_connect = answered_df['connect_sec'].std()
if overall_std_connect > 0:
    agent_stats['connect_z'] = (
        (agent_stats['avg_connect_sec'] - overall_mean_connect) / overall_std_connect
    ).round(2)
else:
    agent_stats['connect_z'] = 0.0

agent_stats = agent_stats[agent_stats['total_contacts'] >= 5].sort_values('connect_z', ascending=False, key=abs)
print(f'Agent stats (agents with ≥5 contacts, {len(agent_stats)} agents):')
print(agent_stats[['agent', 'total_contacts', 'avg_connect_sec', 'connect_z','pct_sla_60s', 'avg_holds', 'avg_attempts', 'pct_multi_attempt']].head(20).to_string(index=False))

In [ ]:
# Multi-connection-attempt analysis
if 'agent_connection_attempts' in df.columns:
    multi = df[df['agent_connection_attempts'] > 1]
    print(f'Contacts with >1 agent connection attempt: {len(multi)} ({len(multi)/len(df)*100:.1f}% of all)')
    print()

    # Per-agent rate
    agent_multi = (
        df.groupby('agent')
        .agg(
            total=('contact_id', 'count'),
            multi=('agent_connection_attempts', lambda x: (x > 1).sum()),
        )
        .reset_index()
    )
    agent_multi['multi_pct'] = (agent_multi['multi'] / agent_multi['total'] * 100).round(1)
    agent_multi = agent_multi[agent_multi['total'] >= 5].sort_values('multi_pct', ascending=False)

    print('Agents with highest multi-attempt rate (≥5 contacts):')
    print(agent_multi.head(10).to_string(index=False))
    print()

    # By queue
    queue_multi = (
        df.groupby('queue')
        .agg(
            total=('contact_id', 'count'),
            multi=('agent_connection_attempts', lambda x: (x > 1).sum()),
        )
        .reset_index()
    )
    queue_multi['multi_pct'] = (queue_multi['multi'] / queue_multi['total'] * 100).round(1)
    queue_multi = queue_multi.sort_values('multi_pct', ascending=False)
    print('Multi-attempt rate by queue:')
    print(queue_multi.to_string(index=False))

In [ ]:
# ACW outliers
acw_by_agent = answered_df.groupby('agent').agg(
    total=('contact_id', 'count'),
    avg_acw_min=('acw_sec', lambda x: x.mean() / 60),
    median_acw_min=('acw_sec', lambda x: x.median() / 60),
).reset_index()
overall_acw_mean = answered_df['acw_sec'].mean() / 60
overall_acw_std = answered_df['acw_sec'].std() / 60
if overall_acw_std > 0:
    acw_by_agent['acw_z'] = ((acw_by_agent['avg_acw_min'] - overall_acw_mean) / overall_acw_std).round(2)
else:
    acw_by_agent['acw_z'] = 0.0
acw_by_agent = acw_by_agent[acw_by_agent['total'] >= 5].sort_values('acw_z', ascending=False, key=abs)

print(f'Overall ACW: mean={overall_acw_mean:.1f} min, std={overall_acw_std:.1f} min')
print()
print('Top ACW outliers by agent (|z| >= 2):')
acw_outliers = acw_by_agent[acw_by_agent['acw_z'].abs() >= 2]
if acw_outliers.empty:
    print('  No agents with ACW z-score >= 2')
else:
    print(acw_outliers[['agent', 'total', 'avg_acw_min', 'acw_z']].to_string(index=False))
print()
print('Highest ACW agents:')
print(acw_by_agent[['agent', 'total', 'avg_acw_min', 'median_acw_min', 'acw_z']].head(10).to_string(index=False))

In [ ]:
# Repeat contacts: same customer phone number within 24h
if 'customer_phone_number' in df.columns:
    df_sorted = df.sort_values(['customer_phone_number', 'initiation_timestamp']).copy()
    df_sorted['prev_time'] = df_sorted.groupby('customer_phone_number')['initiation_timestamp'].shift(1)
    df_sorted['time_since_last'] = (df_sorted['initiation_timestamp'] - df_sorted['prev_time']).dt.total_seconds() / 3600
    df_sorted['is_repeat_24h'] = (df_sorted['time_since_last'] <= 24) & (df_sorted['time_since_last'] > 0)

    repeat_total = df_sorted['is_repeat_24h'].sum()
    repeat_pct = repeat_total / len(df) * 100
    print(f'Repeat contacts (same customer phone within 24h): {repeat_total:,} ({repeat_pct:.1f}% of all)')

    # Customers with most repeats
    repeat_customers = (
        df_sorted[df_sorted['is_repeat_24h']]
        .groupby('customer_phone_number')
        .size()
        .sort_values(ascending=False)
    )
    print(f'Unique customers with repeats: {len(repeat_customers):,}')
    print(f'Customers with 3+ repeat contacts: {(repeat_customers >= 3).sum():,}')
    print(f'Customers with 5+ repeat contacts: {(repeat_customers >= 5).sum():,}')
    print()

    # Repeat rate by queue
    repeat_by_queue = (
        df_sorted.groupby('queue')
        .agg(
            total=('contact_id', 'count'),
            repeats=('is_repeat_24h', 'sum'),
        )
        .reset_index()
    )
    repeat_by_queue['repeat_pct'] = (repeat_by_queue['repeats'] / repeat_by_queue['total'] * 100).round(1)
    repeat_by_queue = repeat_by_queue.sort_values('repeat_pct', ascending=False)
    print('Repeat contact rate by queue:')
    print(repeat_by_queue.to_string(index=False))

    # Repeat rate in last 7d vs baseline
    if len(df_7) > 10:
        df_7_sorted = df_sorted[df_sorted['initiation_timestamp'] >= last_7d]
        repeat_7 = df_7_sorted['is_repeat_24h'].sum()
        repeat_7_pct = repeat_7 / len(df_7_sorted) * 100 if len(df_7_sorted) > 0 else 0
        print(f'\nRepeat rate last 7d: {repeat_7_pct:.1f}% (vs {repeat_pct:.1f}% all-time)')
else:
    print('customer_phone_number column not available')

---
## 6. Volume & Pattern Anomalies

Daily volume with baselines, hour-of-day heatmaps, channel and direction mix shifts.

In [ ]:
# Daily volume with 7d and 30d rolling baselines
daily_vol = df.groupby('date').size().reset_index(name='count')
daily_vol = daily_vol.sort_values('date')
daily_vol['rolling_7d'] = daily_vol['count'].rolling(7, min_periods=1).mean()
daily_vol['rolling_30d'] = daily_vol['count'].rolling(30, min_periods=1).mean()

# Z-score: each day's volume vs. the rolling baseline
daily_vol['vol_std'] = daily_vol['count'].rolling(30, min_periods=7).std()
daily_vol['vol_z'] = ((daily_vol['count'] - daily_vol['rolling_30d']) / daily_vol['vol_std']).round(2)

# Flag volume anomalies
vol_anomalies = daily_vol[daily_vol['vol_z'].abs() >= 2]
print(f'Daily volume anomalies (|z| >= 2): {len(vol_anomalies)} days')
if len(vol_anomalies) > 0:
    print(vol_anomalies[['date', 'count', 'rolling_30d', 'vol_z']].to_string(index=False))
print()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_vol['date'], y=daily_vol['count'], name='Daily Volume', line=dict(color='#3B82F6')))
fig.add_trace(go.Scatter(x=daily_vol['date'], y=daily_vol['rolling_7d'], name='7d Avg', line=dict(color='#F59E0B', dash='dash')))
fig.add_trace(go.Scatter(x=daily_vol['date'], y=daily_vol['rolling_30d'], name='30d Avg', line=dict(color='#EF4444', dash='dot')))
if len(vol_anomalies) > 0:
    fig.add_trace(go.Scatter(
        x=vol_anomalies['date'], y=vol_anomalies['count'],
        mode='markers', name='Anomaly (|z|≥2)',
        marker=dict(color='red', size=10, symbol='x'),
    ))
fig.update_layout(title='Daily Contact Volume with Anomalies', height=500)
fig.show()

In [ ]:
# Day-of-week and hour-of-day patterns
dow_hour = df.groupby(['dow_name', 'hour']).size().reset_index(name='count')
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_hour['dow_name'] = pd.Categorical(dow_hour['dow_name'], categories=dow_order, ordered=True)
pivot = dow_hour.pivot_table(index='dow_name', columns='hour', values='count', fill_value=0)

fig = px.imshow(
    pivot,
    labels=dict(x='Hour', y='Day', color='Contacts'),
    x=pivot.columns,
    y=pivot.index,
    color_continuous_scale='YlOrRd',
    aspect='auto',
)
fig.update_layout(title='Contact Volume Heatmap (Day × Hour)', height=400)
fig.show()

# Day-of-week summary
dow_summary = df.groupby('dow_name').agg(
    total=('contact_id', 'count'),
    avg_connect_sec=('connect_sec', 'mean'),
    abandon_rate=('is_abandoned', 'mean'),
).reset_index()
dow_summary['dow_name'] = pd.Categorical(dow_summary['dow_name'], categories=dow_order, ordered=True)
dow_summary = dow_summary.sort_values('dow_name')
dow_summary['abandon_pct'] = (dow_summary['abandon_rate'] * 100).round(1)
dow_summary['avg_connect_sec'] = dow_summary['avg_connect_sec'].round(1)
print('Day-of-week summary:')
print(dow_summary[['dow_name', 'total', 'avg_connect_sec', 'abandon_pct']].to_string(index=False))

In [ ]:
# Channel and direction mix shifts
if 'channel' in df.columns:
    channel_all = df['channel'].fillna('(none)').value_counts(normalize=True) * 100
    channel_7 = df_7['channel'].fillna('(none)').value_counts(normalize=True) * 100 if len(df_7) > 0 else pd.Series()
    channel_30 = df_30['channel'].fillna('(none)').value_counts(normalize=True) * 100 if len(df_30) > 0 else pd.Series()

    channel_compare = pd.DataFrame({'all_pct': channel_all, '7d_pct': channel_7, '30d_pct': channel_30}).fillna(0)
    channel_compare = channel_compare.sort_values('all_pct', ascending=False)
    print('Channel mix (%):')
    print(channel_compare.round(1).to_string())
    print()

if 'contact_direction' in df.columns:
    dir_all = df['contact_direction'].fillna('(none)').value_counts(normalize=True) * 100
    dir_7 = df_7['contact_direction'].fillna('(none)').value_counts(normalize=True) * 100 if len(df_7) > 0 else pd.Series()
    dir_30 = df_30['contact_direction'].fillna('(none)').value_counts(normalize=True) * 100 if len(df_30) > 0 else pd.Series()

    dir_compare = pd.DataFrame({'all_pct': dir_all, '7d_pct': dir_7, '30d_pct': dir_30}).fillna(0)
    dir_compare = dir_compare.sort_values('all_pct', ascending=False)
    print('Contact direction mix (%):')
    dir_compare = dir_compare.round(1)
    print(dir_compare.to_string())
    print()

# Queue distribution changes
if 'queue' in df.columns:
    queue_all = df['queue'].fillna('(none)').value_counts(normalize=True) * 100
    queue_7 = df_7['queue'].fillna('(none)').value_counts(normalize=True) * 100 if len(df_7) > 0 else pd.Series()
    queue_compare = pd.DataFrame({'all_pct': queue_all, '7d_pct': queue_7}).fillna(0)
    queue_compare['delta'] = (queue_compare['7d_pct'] - queue_compare['all_pct']).round(1)
    queue_compare = queue_compare.sort_values('delta', ascending=False, key=abs)
    print('Queue distribution shifts (7d vs all):')
    print(queue_compare.head(10).to_string())

In [ ]:
# Hour-of-day volume patterns: last 7d vs baseline
hour_all = df.groupby('hour').size().reset_index(name='count_all')
hour_all['pct_all'] = hour_all['count_all'] / hour_all['count_all'].sum() * 100

if len(df_7) > 10:
    hour_7 = df_7.groupby('hour').size().reset_index(name='count_7')
    hour_7['pct_7'] = hour_7['count_7'] / hour_7['count_7'].sum() * 100
    hour_compare = hour_all.merge(hour_7, on='hour', how='outer').fillna(0)
    hour_compare['delta_pct'] = (hour_compare['pct_7'] - hour_compare['pct_all']).round(1)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=hour_compare['hour'], y=hour_compare['pct_all'], name='All Time %', marker_color='#3B82F6', opacity=0.5))
    fig.add_trace(go.Bar(x=hour_compare['hour'], y=hour_compare['pct_7'], name='Last 7d %', marker_color='#EF4444', opacity=0.7))
    fig.update_layout(
        title='Hour-of-Day Volume Share: All Time vs Last 7 Days',
        barmode='group',
        xaxis_title='Hour of Day',
        yaxis_title='% of Daily Volume',
        height=400,
    )
    fig.show()

    # Flag hours with biggest shifts
    hour_compare_sorted = hour_compare.sort_values('delta_pct', ascending=False, key=abs)
    print('Hours with biggest volume share shift (7d vs all):')
    print(hour_compare_sorted[['hour', 'pct_all', 'pct_7', 'delta_pct']].head(10).to_string(index=False))
else:
    print('Not enough data in last 7 days for hourly comparison')

---
## 7. Actionable Findings Summary

Top-ranked anomalies across all categories, with specific callouts.

In [ ]:
findings = []

# --- Wait-time outliers ---
if not outliers_df.empty:
    for _, row in outliers_df.head(10).iterrows():
        direction = 'above' if row['z_score'] > 0 else 'below'
        findings.append({
            'category': 'Wait-Time',
            'finding': f"{row['entity']} ({row['group']}): {row['metric']} is {abs(row['z_score']):.1f}σ {direction} baseline",
            'detail': f"Recent: {row['recent_mean']:.1f}s, Baseline: {row['baseline_mean']:.1f}s, n={row['count']}, period={row['period']}",
            'severity': round(abs(row['z_score']), 1),
        })

# --- Abandonment hotspots ---
for _, row in queue_abandon_all.head(5).iterrows():
    if row['rate'] > 20:
        findings.append({
            'category': 'Abandonment',
            'finding': f"Queue '{row['queue']}' has {row['rate']}% abandonment rate",
            'detail': f"{int(row['abandoned'])} abandoned out of {int(row['total'])} contacts",
            'severity': round(row['rate'] / 10, 1),  # 10% = 1.0 severity
        })

# --- Agent behavior ---
if 'agent_connection_attempts' in df.columns:
    top_multi = agent_multi.head(5) if 'agent_multi' in dir() else pd.DataFrame()
    if not top_multi.empty:
        for _, row in top_multi.iterrows():
            if row['multi_pct'] > 10:
                findings.append({
                    'category': 'Agent Behavior',
                    'finding': f"Agent '{row['agent']}' has {row['multi_pct']}% multi-attempt contacts",
                    'detail': f"{int(row['multi'])} multi-attempt out of {int(row['total'])} contacts",
                    'severity': round(row['multi_pct'] / 10, 1),
                })

# --- ACW outliers ---
if not acw_outliers.empty:
    for _, row in acw_outliers.head(5).iterrows():
        findings.append({
            'category': 'Agent Behavior',
            'finding': f"Agent '{row['agent']}' has ACW z-score {row['acw_z']:.1f}",
            'detail': f"Avg ACW: {row['avg_acw_min']:.1f} min (baseline: {overall_acw_mean:.1f} min), n={int(row['total'])}",
            'severity': round(abs(row['acw_z']), 1),
        })

# --- Volume anomalies ---
if len(vol_anomalies) > 0:
    for _, row in vol_anomalies.head(5).iterrows():
        direction = 'spike' if row['vol_z'] > 0 else 'drop'
        findings.append({
            'category': 'Volume',
            'finding': f"{row['date']} volume {direction}: {int(row['count'])} contacts (z={row['vol_z']:.1f})",
            'detail': f"30d baseline: {row['rolling_30d']:.0f}/day",
            'severity': round(abs(row['vol_z']), 1),
        })

# --- Repeat contacts ---
if 'is_repeat_24h' in dir() or 'is_repeat_24h' in df.columns:
    repeat_pct_val = df_sorted['is_repeat_24h'].mean() * 100 if 'df_sorted' in dir() else 0
    if repeat_pct_val > 5:
        findings.append({
            'category': 'Repeat Contacts',
            'finding': f'{repeat_pct_val:.1f}% of contacts are repeats from the same customer within 24h',
            'detail': 'Potential first-call resolution or callback staffing issue',
            'severity': round(repeat_pct_val / 5, 1),
        })

# Sort by severity and display
findings_df = pd.DataFrame(findings).sort_values('severity', ascending=False)
print(f'=== TOP {len(findings_df)} ANOMALIES (ranked by severity) ===')
print()
for i, (_, row) in enumerate(findings_df.iterrows(), 1):
    print(f'{i}. [{row["category"]}] {row["finding"]}')
    print(f'   {row["detail"]}')
    print(f'   Severity: {row["severity"]}')
    print()

In [ ]:
# Save key datasets for React anomalies page
import json

output_dir = Path('analysis_output')
output_dir.mkdir(exist_ok=True)

# Wait-time outliers
if not outliers_df.empty:
    outliers_df.to_csv(output_dir / 'wait_time_outliers.csv', index=False)

# Queue abandonment
queue_abandon_all.to_csv(output_dir / 'queue_abandonment.csv', index=False)

# Agent stats
agent_stats.to_csv(output_dir / 'agent_stats.csv', index=False)

# Volume anomalies
if len(vol_anomalies) > 0:
    vol_anomalies.to_csv(output_dir / 'volume_anomalies.csv', index=False)

# Findings summary
if not findings_df.empty:
    findings_df.to_csv(output_dir / 'findings_summary.csv', index=False)

print(f'Saved analysis outputs to {output_dir.resolve()}')